# Импорты

Установим все не обходимое для работы

In [1]:
%%capture
!pip install chonkie[all] chonkie optuna qdrant_client mlflow

тут для железа kaggle 

In [ ]:
%%capture
!pip install mlflow

In [8]:
%%capture
!pip install -U cachetools

In [9]:
%%capture
!pip install chonkie[all] chonkie qdrant_client

In [1]:
import os
import subprocess
import shutil
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import uuid
import mlflow
from qdrant_client import QdrantClient
from qdrant_client import models

tqdm.pandas()

# Загрузим и предобработаем датасет

In [2]:
def data_loader(download_path="16a1OP6crVCX1VI6Cukk9eLYYZgd8zAFg", save_path="data"):
    """
    Загружает и распаковывает датасет из Google Drive по указанной ссылке.

    Аргументы:
        download_path (str): ссылка на Google Drive
        save_path (str): путь для сохранения данных

    Возвращает:
        docs - информация о документах и полный текст
        questions - вопрсы для проверки полученных документов
        images - список имен всех изображений
        errors - таблица с кодами ошибок в системе МИС
        roles - таблица с ролями в системе МИС
        terms - таблица с терминами и их рассшировками используемыми в системе МИС
    """
    indices = []
    texts = []

    # очистка и создание папки
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    os.makedirs(save_path, exist_ok=True)

    zip_path = os.path.join(save_path, "dataset.zip")
    dataset_path = os.path.join(save_path, "dataset")

    # скачивание и распаковка
    subprocess.run(["gdown", download_path, "-O", zip_path], check=True)
    subprocess.run(["unzip", "-q", zip_path, "-d", save_path], check=True)

    # список изображений
    images_path = os.path.join(dataset_path, "images")
    images = os.listdir(images_path)

    # загрузка CSV
    questions = pd.read_csv(os.path.join(dataset_path, "questions.csv"))
    docs = pd.read_csv(os.path.join(dataset_path, "texts.csv"))
    errors = pd.read_csv(os.path.join(dataset_path, "errors.csv"))
    roles = pd.read_csv(os.path.join(dataset_path, "roles.csv"))
    terms = pd.read_csv(os.path.join(dataset_path, "terms.csv"))

    # загрузка текстов
    texts_path = os.path.join(dataset_path, "texts")
    for filename in os.listdir(texts_path):
        if filename.endswith(".txt"):
            indices.append(int(filename.split(".")[0]))
            with open(os.path.join(texts_path, filename), encoding="utf-8") as f:
                texts.append(f.read())

    docs = docs.merge(
        pd.DataFrame({"page_id": indices, "text": texts}),
        on="page_id",
        how="left"
    )

    return docs, questions, images, errors, roles, terms

docs, questions, images, errors, roles, terms = data_loader(download_path="1EdLD5-2U0Id_U16YkaEKnu5nxHFGxTir")


FileNotFoundError: [Errno 2] No such file or directory: 'gdown'

In [ ]:
import re
import pandas as pd

def replace_image_name(docs, fieldname, mapping = {}):
    """
    Заменяет uuid изображения на $N, где N — глобальный уникальный номер.
    """

    texts = []
    pattern = r'!\{[^}]*\}[^!\s]+!?|![^!\s]+!|\{\{[^}]+\}\}'
    for _, row in docs.iterrows():
        text = row[fieldname]
        if not isinstance(text, str):
            texts.append(text)
            continue
        images = re.findall(pattern, text)
        for image in images:
            if image not in mapping.values():
                token = f"${len(mapping)}"
                mapping[token] = image
                text = text.replace(image, f" {token}")
            else:
                for t, img in mapping.items():
                    if img == image:
                        text = text.replace(image, f" {t}")
                        break
        texts.append(text.strip())
    docs = docs.copy()
    docs.loc[:, f"clean_{fieldname}"] = texts
    return docs, mapping

def insert_ru_image_name(text, mapping):
    """
    Заменяет $N на "Изображение_N" в тексте.

    Аргументы:
        text: строка
        mapping: словарь $N -> оригинальное имя

    Возвращает:
        text — текст с заменёнными токенами
    """
    def repl(match):
        number = match.group(1)
        return f"Изображение_{number}"
    return re.sub(r"\$(\d+)", repl, text)

In [13]:
from typing import Callable, Tuple

def evaluate(embedding_function : Callable[[str], list[float] | list[list[float]]], questions, client: QdrantClient, collection_name, total_docs, conf_level=0.95) -> Tuple[pd.DataFrame, float, list[float]]:
  """
  Выполняет оценку полученных документов на основе MRR (mean reciprocal rank) метрики

  Аргументы:
      embedding_function: функция, которая будет возращать нам эмбединги
      questions: список вопросов для оценки полученных документов
      collection: база данных, содержащая векторизованных предстваления документов или их чанков
      total_docs: общее количество документов, на их осное расчитывается обратный ранг

  Возвращает:
      df: DataFrame, содержащий список вопросов, ответ, позицию правильного чанка среди всех найденных и id найденных документов
  """
  columns = ["question", "true_chunk", "chunk position", "page position"]
  row_data = []

  for _, row in questions.iterrows():
    question = row["question"]
    chunk_true = row["answer"]

    embedding = embedding_function(question)
    result_points = client.query_points(collection_name, embedding, limit=total_docs).points

    pos_chunk = []

    for i, point in enumerate(result_points):
      if point.payload['page_id'] == row['page_id'] and (point.payload['chunk_id'] in row['answer_chunk_indices']):
        pos_chunk.append(i)

    data_row = [
      question, 
      chunk_true, 
      pos_chunk,
      [page.payload['page_id'] for page in result_points],
      ]

    row_data.append(data_row)

  df = pd.DataFrame(row_data, columns=columns)

  return df

In [14]:
docs = docs.dropna()

In [15]:
docs, mapping = replace_image_name(docs, 'text')
questions, mapping = replace_image_name(questions, 'answer', mapping)


print(insert_ru_image_name(docs.loc[28, "clean_text"], mapping))

h1. Прикрепление через ЕПГУ и по личному заявлению пациента в МО

 Изображение_1

Для работы с заявлениями на прикрепление, полученных с Единого портала государственных услуг (ЕПГУ), назначьте пользователю роли: 
* Оператор рассмотрения заявлений на прикрепление 
* Отклонение заявок на прикрепление

Роль добавляется через блок "Администрирование". Выбираем пункт "Управление пользователями, правами доступа и настройками", находим ФИО пользователя, нажимаем "Редактировать"  и добавляем роли

 Изображение_605

h2. Проверка заявлений с портала ЕПГУ

На главном рабочем столе пользователя в модуле "Медицинские карты" присутствует раздел "Заявления на прикрепление к МО (ЕПГУ)",

 Изображение_606

Перейдите в раздел и откроется журнал заявлений на прикрепление. Выставив фильтры и нажав на кнопку "Найти" отобразятся заявления на прикрепления

 Изображение_607

*Важно!* Сотруднику ответственному за прикрепление должен проверять наличие новых заявлений ежедневно в "Журнал заявлений", так как срок

# Разбиение на чанки


в `field_for_exp` запишем название столбца, куда мы передадим наши чанки 

In [16]:
field_for_exp = 'semantic_clear_text'

In [17]:
from chonkie import SemanticChunker, AutoEmbeddings

model = "deepvk/USER2-base"
embeddings = SentenceTransformer(model)

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
)

def chunking(text):
    chunks = []
    for chunk in chunker.chunk(text):
        chunks.append(chunk.text)
    return chunks

docs.loc[:, field_for_exp] = docs['clean_text'].progress_apply(chunking)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

100%|██████████| 172/172 [03:01<00:00,  1.06s/it]


# Добавим информацию из docs в question

Надо несколько полей добавить, а именно `page_id` и наши чанки

`answer_chunk_indices` - будет находится положение наших чанков, дело в том что из-за чанкования это шутка динамическая, поэтому придется тратить время на расчеты

In [18]:
def get_chunks_and_page_id(row, docs):
    doc_row = docs[docs['title'] == row['title']]
    if not doc_row.empty:
        return pd.Series({
            field_for_exp: doc_row.iloc[0][field_for_exp],
            'page_id': doc_row.iloc[0]['page_id']
        })
    return pd.Series({field_for_exp: [], 'page_id': None})

questions[[field_for_exp, 'page_id']] = questions.apply(
    lambda row: get_chunks_and_page_id(row, docs), axis=1
)

In [19]:
def get_answer_chunk_indices(answer: str, chunks):
    indices = []
    answer_words = set(re.split(r"\s+", answer))
    for i in range(len(chunks)):
        for j in range(i, len(chunks) + 1):
            merged = "".join(chunks[i:j]) if i != j else chunks[i]
            merged_words = set(re.split(r"\s+", merged))
            if answer_words == (merged_words & answer_words):
                indices.append([i, j])
    if len(indices) == 0:
        return indices
    labels = min(indices, key=lambda x: x[1] - x[0])
    return list(range(labels[0], labels[1] + 1))

questions['answer_chunk_indices'] = questions.progress_apply(
    lambda row: get_answer_chunk_indices(row['clean_answer'], row[field_for_exp]),
    axis=1)

100%|██████████| 24/24 [00:03<00:00,  6.37it/s]


# Дальше проводим эксперимент

In [20]:
os.environ['MLFLOW_TRACKING_USERNAME'] = 'admin'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'SuperSecurePassword'

# Адрес mlflow
mlflow.set_tracking_uri("http://109.207.175.83:5000/")
mlflow.set_experiment("alpha_team")

# Клиент Qdrant
client = QdrantClient(
    url="http://109.207.175.83:6333",
    api_key="f860f5aef884d8e282f1a8cc9ef71557o",
)

/tmp/ipykernel_89/4130787539.py:9: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [21]:
collection_name=str(uuid.uuid1())

client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=embeddings.get_sentence_embedding_dimension(),  # Vector size is defined by used model
        distance=models.Distance.COSINE,
        multivector_config=models.MultiVectorConfig(
            comparator=models.MultiVectorComparator.MAX_SIM
        ),
    ),
)


with mlflow.start_run(run_name="multi-vector + chonkie + vk_embeds"):
    mlflow.log_param("model", model)
    mlflow.log_param("model_params", embeddings.__dict__)
    mlflow.log_param("collection_name", collection_name)
    mlflow.log_param("chunker_name", str(chunker.__class__))
    mlflow.log_param("chunker_params", chunker.__dict__)

    chunk_lengths = []
    points = []

    for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
        texts = row[field_for_exp]

        for chunk_id, chunk in enumerate(texts):
            points.append(models.PointStruct(
                id=len(points) + 1
                , vector=[
                    embeddings.encode(chunk),
                    embeddings.encode(chunk, prompt_name='search_document'),
                    embeddings.encode(chunk, prompt_name='search_query')
                ]
                , payload={"page_id": row["page_id"], "chunk": chunk, "chunk_id": chunk_id}
                ))

            chunk_lengths.append(len(chunk))


    mlflow.log_metric("total_chunks", len(chunk_lengths))
    mlflow.log_metric("min_chunk_length", min(chunk_lengths))
    mlflow.log_metric("max_chunk_length", max(chunk_lengths))
    mlflow.log_metric("avg_chunk_length", sum(chunk_lengths)/len(chunk_lengths))

    client.upsert(
        collection_name=collection_name,
        points=points
    )

    def embeddings_function(question: str) -> list[list[float]]:
        return [
            embeddings.encode(question).tolist(),
            embeddings.encode(question, prompt_name='search_document').tolist(),
            embeddings.encode(question, prompt_name='search_query').tolist()
        ]

    # --- Оценка разбиения ---
    eval_df= evaluate(embeddings_function, questions, client, collection_name, len(docs))
    # Сохраняем таблицу как артефакт
    mlflow.log_table(eval_df, "eval_df.json")
    # # Логируем метрику MRR и доверительный интервал
    # mlflow.log_metric("MRR", mrr)
    # mlflow.log_metric("MRR_lower", float(mrr_ci[0]))
    # mlflow.log_metric("MRR_upper", float(mrr_ci[1]))

100%|██████████| 172/172 [04:42<00:00,  1.64s/it]


🏃 View run multi-vector + chonkie + vk_embeds at: http://109.207.175.83:5000/#/experiments/2/runs/b55a4bb1542d4033b6936f4c659bdc80
🧪 View experiment at: http://109.207.175.83:5000/#/experiments/2
